# Day 1 — Explore MARTA

**Estimated time:** ~2.5 hours  •  **Sections:** 3

## Learning objectives

By the end of this notebook you will be able to:

- Load a real transit agency's GTFS (General Transit Feed
  Specification) feed with pandas.
- Recognize a common schema surprise — MARTA doesn't use the
  `location_type` parent-station convention — and know how to handle
  it.
- Produce a summary of MARTA's rail system (stations, lines, stops).

## Table of contents

1. [Section 1 — Load the GTFS feed](#section-1) (~60 min)
2. [Section 2 — Filter to rail, summarize the system](#section-2) (~60 min)
3. [Section 3 — Preview Day 2](#section-3) (~30 min)

---

## Setup

Run the next cell to bring up the environment.


In [ ]:
# Setup — run this first. In Colab: clones the repo and installs deps.
# Locally: no-op (the conda env handles it).
import os, sys, urllib.request
try:
    import google.colab  # noqa: F401
    if not os.path.exists('/tmp/colab_bootstrap.py'):
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/arctic-gsu/arctic-scisynth-2026-rwd-public/main/setup/colab_bootstrap.py',
            '/tmp/colab_bootstrap.py',
        )
    sys.path.insert(0, '/tmp')
    from colab_bootstrap import bootstrap
    bootstrap()
except ImportError:
    pass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'raw').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'raw').is_dir(), (
    f'❌ Could not find data/raw from {Path.cwd()}. '
    'If you are in Colab, re-run this cell — the bootstrap may not have completed.'
)
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
CHECKPOINTS = ROOT / 'data' / 'checkpoints'

# MARTA rail-adjacent constants — referenced throughout the week.
MARTA_NTD_ID = 40022  # 5-digit NTD ID. Older docs use '4022' — that is wrong.
MODE_NAMES = {'HR': 'Heavy Rail', 'MB': 'Bus', 'SR': 'Streetcar',
              'DR': 'Demand Response'}
STADIUM_ADJACENT = ['sec district', 'vine city']   # where fans alight
STADIUM_AREA = STADIUM_ADJACENT + ['five points']  # + transfer hub

print(f'✅ Imports ready. Working from {ROOT}')


<a id="section-1"></a>
## Section 1 — Load the GTFS feed

MARTA — like every major transit agency — publishes a **GTFS feed**: a
zip of CSV-like text files describing stops, routes, trips, and
schedules. The same format powers Google Maps transit directions. The
skills you build today transfer to any transit system in the world.

We committed the feed locally so you don't need network access; it
lives at `data/raw/marta_gtfs.zip`.


In [ ]:
import zipfile

with zipfile.ZipFile(RAW / 'marta_gtfs.zip') as zf:
    print('Files inside the GTFS zip:')
    for name in sorted(zf.namelist()):
        print(f'  {name}')


Each `.txt` file is a CSV. The most important for today is
**`stops.txt`** — it has one row per bus stop or rail station, with
columns for the stop ID, name, latitude, and longitude.


In [ ]:
with zipfile.ZipFile(RAW / 'marta_gtfs.zip') as zf:
    stops = pd.read_csv(zf.open('stops.txt'))

print(f'stops.txt shape: {stops.shape}')
stops.head()


### 🎯 Your Turn! How many stops and how many unique stop names?


In [ ]:
# Count rows and unique stop names.
n_stops = ???
n_unique_names = ???
print(f'Total stop rows: {n_stops}')
print(f'Unique stop names: {n_unique_names}')


<details><summary>💡 Click for solution</summary>

```python
n_stops = len(stops)
n_unique_names = stops['stop_name'].nunique()
```

</details>


### ✅ Checkpoint


In [ ]:
assert n_stops > 1000, f'❌ Expected >1000 rows in stops.txt; got {n_stops}.'
assert n_unique_names < n_stops, (
    '❌ Each station should have multiple platforms, so unique names '
    'should be smaller than total rows. If they match, re-check n_unique_names.'
)
print(f'✅ {n_stops} rows, {n_unique_names} unique station/stop names.')


### 🔄 Recovery point

If Section 1 went sideways, run this cell and continue to Section 2.


In [ ]:
import zipfile
with zipfile.ZipFile(RAW / 'marta_gtfs.zip') as zf:
    stops = pd.read_csv(zf.open('stops.txt'))
    routes = pd.read_csv(zf.open('routes.txt'))
print(f'✅ Recovered: stops ({len(stops)} rows) and routes ({len(routes)} rows).')


---

<a id="section-2"></a>
## Section 2 — Filter to rail stations, summarize the system

MARTA runs **heavy rail** (subway-style) and **buses**. For the World
Cup analysis we care about rail — fans move between hotels, the
stadium, and the airport on the Red/Gold/Blue/Green lines, not on
local buses. Let's find just the rail stations.

The standard GTFS way to do this is to filter `stops.txt` by
`location_type == 1` (the "parent station" flag). Let's try that and
see what happens.


In [ ]:
# Try the textbook GTFS filter: location_type == 1 means "parent station".
n_parent = (stops['location_type'] == 1).sum()
print(f'location_type == 1 returns {n_parent} rows.')
print(f"location_type is NaN for {stops['location_type'].isna().sum()} of {len(stops)} rows.")


**Surprise.** MARTA's GTFS feed doesn't populate `location_type`
at all — the textbook filter returns **zero rows**. Real-world data
drift is a fact of life when working with open transit feeds.

The correct path — joining `stop_times` → `trips` → `routes` on
`route_type == 1` (heavy rail) — is a non-trivial four-table join.
Instead of working through that join in your first session with
pandas, we've pre-built the result: **`data/processed/rail_stops.csv`**.
Load it and continue.


In [ ]:
# 🎯 Your Turn! Load the pre-staged rail_stops.csv file.
rail_stops = pd.read_csv(???)
rail_stops.head()


<details><summary>💡 Click for solution</summary>

```python
rail_stops = pd.read_csv(PROCESSED / 'rail_stops.csv')
```

</details>


### System summary

Now produce a short summary of MARTA's rail system. **Four lines** (Red,
Gold, Blue, Green) share **38 stations** across the metro core.


In [ ]:
# Number of lines — from the routes table, filtered to heavy rail.
rail_routes = routes[routes['route_type'] == 1]
print('Rail routes:')
print(rail_routes[['route_short_name', 'route_long_name']].to_string(index=False))

print(f"\nTotal rail stations: {len(rail_stops)}")
print(f"Total rail lines: {len(rail_routes)}")
print(f"Total stops (all modes): {len(stops)}")


One more naming note: the station formerly known as **"GWCC/CNN
Center"** was renamed **"SEC District Station"** in the current GTFS
feed. It's the closest rail station to Mercedes-Benz Stadium
(~0.32 km away). Older MARTA documentation still uses the old name;
the GTFS feed is the source of truth for this week.


### 🎯 Your Turn! Find the stations closest to 'SEC' in their name.


In [ ]:
# Use .str.contains to find stations whose name contains "SEC".
# Hint: pass case=False, na=False.
sec_matches = rail_stops[rail_stops['station_name'].str.contains(???)]
sec_matches


<details><summary>💡 Click for solution</summary>

```python
sec_matches = rail_stops[rail_stops['station_name'].str.contains('SEC', case=False, na=False)]
```

</details>


### ✅ Checkpoint


In [ ]:
assert len(rail_stops) == 38, (
    f'❌ Expected 38 rail stations; got {len(rail_stops)}. '
    'Check that rail_stops.csv loaded correctly.'
)
assert len(sec_matches) >= 1, (
    "❌ Expected at least one 'SEC District' station. "
    'Check your .str.contains call.'
)
print(f'✅ 38 stations, {len(sec_matches)} SEC match(es).')


### 🔄 Recovery point

Run the next cell to reload a clean `rail_stops` DataFrame and continue
to Section 3.


In [ ]:
rail_stops = pd.read_csv(PROCESSED / 'rail_stops.csv')
print(f'✅ Loaded {len(rail_stops)} rail stations. Ready for Section 3.')


---

<a id="section-3"></a>
## Section 3 — Save today's work, preview Day 2

Save your rail station list as today's **checkpoint file**. Tomorrow's
notebook will start by loading this file, so no matter what happens
overnight, Day 2 can pick up from a known-good Day 1 output.


In [ ]:
out = rail_stops[['station_name', 'lat', 'lon']].copy()
out.to_csv(CHECKPOINTS / 'day1_output_stops.csv', index=False)
print(f'✅ Wrote {len(out)} stations to data/checkpoints/day1_output_stops.csv')


### Preview: what's coming in Day 2

Tomorrow you'll work with **`data/processed/station_frequencies.csv`**,
a per-station trains-per-hour file. Here's a preview — notice the
messiness. Cleaning it is part of Day 2's work.


In [ ]:
freq_preview = pd.read_csv(PROCESSED / 'station_frequencies.csv')
print(f'Shape: {freq_preview.shape}')
freq_preview.head(10)


You should see:

- **Inconsistent station names** (mixed capitalization, stray
  `STATION` suffix)
- **Some null values** in `offpeak_trains_per_hour`
- **Duplicate rows** (a couple of stations listed twice)

Don't clean it yet — that's Section 3 of Day 2.

## What's next?

Tomorrow in **`day2_cleaning_ridership.ipynb`** you'll:

1. Level up from the morning's cleaning session: load the full raw
   NTD multi-tab Excel workbook.
2. Plot MARTA's ridership trend since 2019 and discuss the
   faregate-data-quality anomaly.
3. Clean this messy `station_frequencies.csv` and identify the
   stadium-adjacent stations.

See you then. 👋
